# Binaryzacja

### Zadanie domowe - binaryzacja adaptacyjna w oknach z interpolacją.

Pokazana w ramach podstawowej części ćwiczenia binaryzacja adaptacyjna działa dobrze, ale jest dość złożona obliczeniowo (choć oczywiście należy mieć świadomość, że implementację można zoptymalizować i wyeliminować pewne powtarzające się obliczenia).
Zbliżone rozwiązanie można również realizować w nieco innym wariancie - w oknach.
Ogólna idea jest następująca: wejściowy obraz dzielimy na nienachodzące (rozłączne) okna - wygodnie jest założyć, że są one kwadratowe i o rozmiarze będącym potęgą liczby 2.
W każdym z okien obliczamy próg - niech to będzie średnia i stosujemy do binaryzacji lokalnej.
Jak nietrudno się domyślić efekt nie będzie dobry, gdyż na granicach okien wystąpią artefakty.
Aby je wyeliminować należy zastosować interpolację, co zostanie szczegółowo opisane poniżej.
Warto zaznaczyć, że podobny mechanizm interpolacji stosowany jest w poznanym wcześniej algorytmie CLAHE.
Zainteresowane osoby odsyłam do artykułu na [Wikipedii](https://en.wikipedia.org/wiki/Adaptive_histogram_equalization) oraz do artykułu o metodzie CLAHE - Zuiderveld, Karel. “Contrast Limited Adaptive Histograph Equalization.” Graphic Gems IV. San Diego: Academic Press Professional, 1994. 474–485.



Na początek zaimplementujemy wariant metody bez interpolacji:
1. Wczytaj obraz _rice.png_.
2. W dwóch pętlach `for`, dla okien o ustalonym wymiarze $W$ (potęga 2), oblicz średnią:
- pętle powinny mieć krok $W$,
- wynik (tj. średnie) należy zapisać w pomocniczej tablicy,
- przydatny operator to `//` - dzielenie całkowitoliczbowe (*floor division*).

3. W kolejnych dwóch pętlach `for` (tym razem o kroku 1) przeprowadź binaryzację z wyznaczonymi progami.
   Tu oczywiście należy się sprytnie odwołać do wyników z tablicy pomocniczej.
   Wyświetl wyniki - czy jest on poprawny?
   Podpowiedź - błędy lepiej widać dla mniejszego rozmiaru okna (np. 16 x 16).

In [ ]:
"""
Przetwarzanie obrazu: Proces binaryzacji obrazu przy użyciu interpolacji bilinearną.
Kod wykonuje binaryzację obrazu metodą bez interpolacji oraz metodą z interpolacją bilinearną.
"""

import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

if not os.path.exists("rice.png"):
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/rice.png --no-check-certificate
        
rice = cv2.imread("rice.png", cv2.IMREAD_GRAYSCALE)

W = 16

(X, Y) = rice.shape

rice_bin_no_interp = np.zeros(rice.shape)
prog = np.zeros((X // W + 1, Y // W + 1))
for i in range(0, X, W):
    for j in range(0, Y, W):
        i_end = min(i + W, X)
        j_end = min(j + W, Y)
        otoczenie = rice[i:i_end, j:j_end]
        prog[i // W][j // W] = np.mean(otoczenie)

for i in range(X):
    for j in range(Y):
        rice_bin_no_interp[i][j] = (rice[i][j] > prog[i // W][j // W]).astype("uint8")

rice_bin_interp = np.zeros(rice.shape)

for i in range(X):
    for j in range(Y):
        window_i = i // W
        window_j = j // W
        rel_i = (i % W) / W
        rel_j = (j % W) / W
        has_right = window_j + 1 < prog.shape[1]
        has_bottom = window_i + 1 < prog.shape[0]
        has_bottom_right = has_right and has_bottom
        threshold = 0
        weight_top_left = (1 - rel_i) * (1 - rel_j)
        threshold += weight_top_left * prog[window_i, window_j]
        if has_right:
            weight_top_right = (1 - rel_i) * rel_j
            threshold += weight_top_right * prog[window_i, window_j + 1]
        if has_bottom:
            weight_bottom_left = rel_i * (1 - rel_j)
            threshold += weight_bottom_left * prog[window_i + 1, window_j]
        if has_bottom_right:
            weight_bottom_right = rel_i * rel_j
            threshold += weight_bottom_right * prog[window_i + 1, window_j + 1]
        rice_bin_interp[i][j] = (rice[i][j] > threshold).astype("uint8")

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(rice, "gray", vmin=0, vmax=256)
axs[0].set_title("Obraz oryginalny")
axs[0].axis("off")

axs[1].imshow(rice_bin_no_interp, "gray")
axs[1].set_title("Bez interpolacji")
axs[1].axis("off")

axs[2].imshow(rice_bin_interp, "gray")
axs[2].set_title("Z interpolacją bilinearną")
axs[2].axis("off")

plt.tight_layout()
plt.show()

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
fragment_y, fragment_x = 100, 100
fragment_size = 50

axs[0].imshow(rice[fragment_y:fragment_y + fragment_size, fragment_x:fragment_x + fragment_size], "gray")
axs[0].set_title("Fragment oryginalny")
axs[0].axis("off")

axs[1].imshow(rice_bin_no_interp[fragment_y:fragment_y + fragment_size, fragment_x:fragment_x + fragment_size], "gray")
axs[1].set_title("Fragment bez interpolacji")
axs[1].axis("off")

axs[2].imshow(rice_bin_interp[fragment_y:fragment_y + fragment_size, fragment_x:fragment_x + fragment_size], "gray")
axs[2].set_title("Fragment z interpolacją")
axs[2].axis("off")

plt.tight_layout()
plt.show()

4. Rozwiązaniem problemu artefaktów na obrazie jest zastosowanie interpolacji.
   Próg binaryzacji dla danego okna wyliczany jest na podstawie progów z sąsiednich okien.
   ![Ilustracja koncepcji interpolacji](https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/clahe_tile_interpolation.png)

   Koncepcja została przedstawiona na powyższym rysunku.
   Możliwe są 3 przypadki:
   - piksel leży w rogach obrazu (kolor czerwony) - wtedy za próg przyjmuje się wartość średniej obliczonej dla danego okna,
   - piksel leży na krawędzi obrazu (kolor zielony) - wtedy za próg przyjmuje się wartość obliczoną na podstawie średnich z dwóch sąsiednich okien,
   - piksel leży w środku (kolor fioletowy) - wtedy próg jest obliczany na podstawie 4 sąsiednich okien.

   Uwaga. Proszę zwrócić uwagę, że sprawa jest dość złożona.
   Obraz dzielimy na okna (dla nich liczymy średnią) i następnie każde z okien "wirtualnie" na cztery sub-okna (linie przerywane).
   To ułatwia znalezienie środków okien (czarne kwadraty), które są wykorzystywane w interpolacji.

5. Implementujemy interpolację.
   Potrzebujemy do tego znać progi (jeden, dwa lub cztery), ale dla przejrzystości obliczeń lepiej zawsze przyjąć cztery oraz odległości od rozważnego piksela do środka sąsiednich okien (też w ogólnym przypadku 4):
   - całość sprowadza się do określania pozycji piksela,
   - na początek rozważmy przypadek czterech narożników (kolor czerwony na rysunku) - trzeba napisać `if`, który je wyznaczy,
   - warto sprawdzić, czy nie popełniliśmy błędu i np. tymczasowo do obrazu wynikowego w tym miejscu przypisać wartość 255. Efekt powinien być taki, że widoczne będą tylko narożniki.
   - drugi przypadek do brzegi (kolor zielony) - postępujemy podobnie jak przy narożnikach, przy czym osobno wydzielamy brzegi pionowe i poziome. Tu też warto sobie obrazek "pokolorować".
   - na koniec wyznaczamy piksele w środku.
   - analizując poprawność proszę zwrócić uwagę na to, żeby nie było przerw pomiędzy obszarami.
   - mając podział możemy dla każdego z obszarów wyliczyć cztery progi ($t11, t12, t21, t22$):
        - dla narożników wartość ta będzie identyczna i wynosi po prostu `t11 =t[jT][iT]`, gdzie `iT=i//W` oraz `jT=j//W`.
          Uwaga. Proszę używać indeksów tymczasowych $jT,iT$, gdyż będą potrzebne w dalszych obliczeniach.
        - dla brzegów pionowych występują dwie wartości: okno bieżące i sąsiednie.
          Wyznaczenie współrzędnej poziomej jest proste (jak dla narożników).
          Nad współrzędną pionową trzeba się chwilę zastanowić - aby nie rozważać wielu przypadków można od bieżącej współrzędnej odjąć połowę rozmiaru okna i dopiero później wykonać dzielenie przez rozmiar okna.
          W ten sposób otrzymujemy indeks okna o mniejszej współrzędnej.
          Indeks drugiego uzyskamy dodając 1.
          Proszę się zastanowić dlaczego to działa - najlepiej to sobie rozrysować.
        - dla brzegów poziomych należy postąpić analogicznie,
        - obliczenia dla obszaru wewnątrz powinny być już oczywiste.
   - kolejny krok to wyliczenie odległości pomiędzy rozważanym pikselem, a czterema środkami.
     Przykładowo dla osi X wygląda to następująco: `dX1 = i - W/2 - iT*W` oraz `dX2 = (iT+1)*W - i-W/2`.
     Dla osi Y analogicznie.
     Ponownie proszę się zastanowić dlaczego to jest poprawne - najlepiej to sobie narysować.
   - ostatni krok to interpolacja dwuliniowa.
     Wykonamy ją w trzech krokach:
     - interpolacja w osi X dla dwóch górnych okien - sprowadza się ona do średniej ważonej pomiędzy wartościami $t11$ i $t12$, przy czym wagi to odpowiednio $dX2/W$ i $dX1/W$.
       Ponownie na podstawie rysunku proszę to przemyśleć.
     - interpolacja w osi X dla dolnych okien jest analogiczna,
     - interpolacja w osi Y również jest analogiczna, z tym, że wejściem są dwa wyniki interpolacji w poziomie.

6. "Kropka nad i" to oczywiście binaryzacja z wyznaczonym poprzez interpolację progiem - proszę dobrać rozmiar okna.
7. Na koniec proszę porównać na wspólnym rysunku poznane metody binaryzacji:
- Otsu,
- lokalna na podstawie średniej,
- lokalna Sauvoli,
- lokalna w oknach bez interpolacji,
- lokalna w oknach z interpolacją.

Proszę pod porównaniem, w osobnej sekcji *markdown*, krótko skomentować uzyskane wyniki.

In [ ]:
"""
Binaryzacja adaptacyjna - hybrydowe podejście

Ten skrypt implementuje różne metody binaryzacji obrazu, ze szczególnym 
uwzględnieniem binaryzacji adaptacyjnej w oknach z interpolacją. 
Porównane są następujące metody:

1. Metoda Otsu (globalna)
2. Binaryzacja lokalna na podstawie średniej
3. Binaryzacja lokalna metodą Sauvoli
4. Binaryzacja adaptacyjna w oknach bez interpolacji
5. Binaryzacja adaptacyjna w oknach z interpolacją bilinearną

Data: 2025-03-27
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import urllib.request
import ssl

def pobierz_obraz(nazwa_pliku="rice.png", url=None):
    """
    Pobiera obraz z dysku lub z internetu, jeśli nie istnieje lokalnie
    """
    if not os.path.exists(nazwa_pliku):
        if url is None:
            url = "https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/rice.png"
        print(f"Pobieram obraz {nazwa_pliku} z internetu...")
        
        ssl._create_default_https_context = ssl._create_unverified_context
        urllib.request.urlretrieve(url, nazwa_pliku)
        print(f"Zapisano jako {nazwa_pliku}")
    
    obraz = cv2.imread(nazwa_pliku, cv2.IMREAD_GRAYSCALE)
    if obraz is None:
        raise FileNotFoundError(f"Nie udało się wczytać obrazu {nazwa_pliku}")
    
    return obraz


def wizualizacja_porownawcza(obrazy, tytuly, figsize=(15, 10)):
    """
    Wyświetla porównanie obrazów w siatce 2x3
    """
    fig, axs = plt.subplots(2, 3, figsize=figsize)
    axs = axs.flatten()
    
    for i, (obraz, tytul) in enumerate(zip(obrazy, tytuly)):
        if i < len(axs):
            axs[i].imshow(obraz, cmap='gray')
            axs[i].set_title(tytul)
            axs[i].axis('off')
    
    plt.tight_layout()
    plt.show()


def wizualizacja_fragmentow(obrazy, tytuly, fragment_y=100, fragment_x=100, fragment_size=50):
    """
    Wyświetla powiększone fragmenty obrazów dla lepszego porównania szczegółów
    """
    fig, axs = plt.subplots(1, len(obrazy), figsize=(15, 5))
    
    for i, (obraz, tytul) in enumerate(zip(obrazy, tytuly)):
        fragment = obraz[fragment_y:fragment_y+fragment_size, fragment_x:fragment_x+fragment_size]
        axs[i].imshow(fragment, cmap='gray')
        axs[i].set_title(f"Fragment: {tytul}")
        axs[i].axis('off')
    
    plt.tight_layout()
    plt.show()

def binaryzacja_otsu(obraz):
    """
    Implementacja binaryzacji globalnej metodą Otsu
    
    Parametry:
    obraz - obraz wejściowy w skali szarości
    
    Zwraca:
    obraz binarny
    """
    _, wynik = cv2.threshold(obraz, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return wynik


def binaryzacja_lokalna_srednia(obraz, window_size=15):
    """
    Implementacja binaryzacji lokalnej na podstawie średniej
    
    Parametry:
    obraz - obraz wejściowy w skali szarości
    window_size - rozmiar okna
    
    Zwraca:
    obraz binarny
    """
    lokalne_srednie = cv2.blur(obraz.astype(np.float32), (window_size, window_size))
    wynik = np.zeros_like(obraz, dtype=np.uint8)
    wynik[obraz > lokalne_srednie] = 255
    return wynik


def binaryzacja_sauvoli(obraz, window_size=15, k=0.15, R=128):
    """
    Implementacja binaryzacji metodą Sauvoli
    
    Parametry:
    obraz - obraz wejściowy w skali szarości
    window_size - rozmiar okna
    k - parametr metody Sauvoli (zwykle 0.15-0.5)
    R - zakres dynamiki obrazu (typowo 128 dla obrazów 8-bitowych)
    
    Zwraca:
    obraz binarny
    """
    obraz_float = obraz.astype(np.float32)
    lokalne_srednie = cv2.blur(obraz_float, (window_size, window_size))
    
    lokalne_srednie_kw = cv2.blur(obraz_float**2, (window_size, window_size))
    lokalne_odch_std = np.sqrt(np.maximum(lokalne_srednie_kw - lokalne_srednie**2, 0))
    
    progi = lokalne_srednie * (1 + k * ((lokalne_odch_std / R) - 1))
    
    wynik = np.zeros_like(obraz, dtype=np.uint8)
    wynik[obraz > progi] = 255
    return wynik


def binaryzacja_okienkowa_bez_interpolacji(obraz, W):
    """
    Implementacja binaryzacji adaptacyjnej w oknach bez interpolacji
    
    Parametry:
    obraz - obraz wejściowy w skali szarości
    W - rozmiar okna (najlepiej potęga liczby 2)
    
    Zwraca:
    obraz binarny, tablica progów dla okien
    """
    wysokosc, szerokosc = obraz.shape
    
    num_okien_y = (wysokosc + W - 1) // W
    num_okien_x = (szerokosc + W - 1) // W
    
    progi = np.zeros((num_okien_y, num_okien_x), dtype=np.float32)
    for i in range(0, wysokosc, W):
        for j in range(0, szerokosc, W):
            i_end = min(i + W, wysokosc)
            j_end = min(j + W, szerokosc)
            okno = obraz[i:i_end, j:j_end]
            progi[i // W, j // W] = np.mean(okno)
    
    wynik = np.zeros_like(obraz, dtype=np.uint8)
    for i in range(wysokosc):
        for j in range(szerokosc):
            if i // W < num_okien_y and j // W < num_okien_x:
                prog = progi[i // W, j // W]
                wynik[i, j] = 255 if obraz[i, j] > prog else 0
    
    return wynik, progi


def binaryzacja_okienkowa_z_interpolacja(obraz, W):
    """
    Implementacja binaryzacji adaptacyjnej w oknach z interpolacją bilinearną
    
    Parametry:
    obraz - obraz wejściowy w skali szarości
    W - rozmiar okna (najlepiej potęga liczby 2)
    
    Zwraca:
    obraz binarny
    """
    wysokosc, szerokosc = obraz.shape
    
    wynik_bez_interp, progi = binaryzacja_okienkowa_bez_interpolacji(obraz, W)
    
    num_okien_y, num_okien_x = progi.shape
    wynik = np.zeros_like(obraz, dtype=np.uint8)
    
    def oblicz_prog_z_interpolacja(y, x):
        center_y = y // W
        center_x = x // W
        
        if (y < W/2 and x < W/2) or \
           (y < W/2 and x >= szerokosc - W/2) or \
           (y >= wysokosc - W/2 and x < W/2) or \
           (y >= wysokosc - W/2 and x >= szerokosc - W/2):
            return progi[y // W, x // W]
        
        rel_y = (y % W) / W
        rel_x = (x % W) / W
        
        i_left = max(0, (y - W//2) // W)
        i_right = min(num_okien_y - 1, i_left + 1)
        j_top = max(0, (x - W//2) // W)
        j_bottom = min(num_okien_x - 1, j_top + 1)
        
        if y < W/2 or y >= wysokosc - W/2:  
            t1 = progi[y // W, j_top]
            t2 = progi[y // W, j_bottom]
            return t1 * (1 - rel_x) + t2 * rel_x
            
        elif x < W/2 or x >= szerokosc - W/2: 
            t1 = progi[i_left, x // W]
            t2 = progi[i_right, x // W]
            return t1 * (1 - rel_y) + t2 * rel_y
            
        else:  
            t11 = progi[i_left, j_top]      
            t12 = progi[i_left, j_bottom]   
            t21 = progi[i_right, j_top]     
            t22 = progi[i_right, j_bottom]  
            
            t_top = t11 * (1 - rel_x) + t12 * rel_x
            t_bottom = t21 * (1 - rel_x) + t22 * rel_x
            return t_top * (1 - rel_y) + t_bottom * rel_y
    
    for i in range(wysokosc):
        for j in range(szerokosc):
            prog = oblicz_prog_z_interpolacja(i, j)
            wynik[i, j] = 255 if obraz[i, j] > prog else 0
    
    return wynik


def main():
    obraz = pobierz_obraz("rice.png")
    
    W = 16  
    
    wynik_otsu = binaryzacja_otsu(obraz)
    wynik_lokalna_srednia = binaryzacja_lokalna_srednia(obraz, W)
    wynik_sauvoli = binaryzacja_sauvoli(obraz, W, k=0.15, R=128)
    wynik_okna_bez_interp, _ = binaryzacja_okienkowa_bez_interpolacji(obraz, W)
    wynik_okna_z_interp = binaryzacja_okienkowa_z_interpolacja(obraz, W)
    
    obrazy = [obraz, wynik_otsu, wynik_lokalna_srednia, 
              wynik_sauvoli, wynik_okna_bez_interp, wynik_okna_z_interp]
    
    tytuly = ["Obraz oryginalny", "Metoda Otsu", "Lokalna (średnia)",
              "Lokalna Sauvoli", "Okna bez interpolacji", "Okna z interpolacją"]
    
    print("Porównanie różnych metod binaryzacji:")
    wizualizacja_porownawcza(obrazy, tytuly)
    
    print("Porównanie fragmentów obrazów (widoczność artefaktów):")
    wizualizacja_fragmentow([obraz, wynik_okna_bez_interp, wynik_okna_z_interp], 
                           ["Oryginalny", "Bez interpolacji", "Z interpolacją"])


if __name__ == "__main__":
    main()
    
    print("\nAnaliza wyników:")
    print("""
Porównanie metod binaryzacji:

1. Metoda Otsu:
   - Metoda globalna, szybka, ale nie radzi sobie z nierównomiernym oświetleniem
   - Widać, że ziarna w jaśniejszych obszarach są dobrze wykryte, ale w ciemniejszych regionach są pomijane

2. Binaryzacja lokalna na podstawie średniej:
   - Lepiej radzi sobie z nierównomiernym oświetleniem
   - Jednak może generować szum w jednolitych obszarach

3. Binaryzacja metodą Sauvoli:
   - Uwzględnia lokalne odchylenie standardowe, co redukuje szumy
   - Bardziej odporna na zmiany oświetlenia i lepiej zachowuje detale

4. Binaryzacja w oknach bez interpolacji:
   - Widoczne wyraźne artefakty na granicach okien w postaci "schodków"
   - Efekt ten jest szczególnie widoczny na powiększonym fragmencie

5. Binaryzacja w oknach z interpolacją:
   - Eliminuje problem artefaktów na granicach
   - Zapewnia płynne przejścia między oknami
   - Daje naturalniejsze wyniki binaryzacji przy podobnej efektywności obliczeniowej jak metoda bez interpolacji

Metoda z interpolacją bilinearną stanowi dobry kompromis między jakością a złożonością obliczeniową,
ponieważ wykorzystuje progi obliczone tylko dla ograniczonej liczby okien, a nie dla otoczenia każdego piksela.
    """)
    